---
title: Relative-strength days during corrections
execute:
  enabled: true
---

`q.indicator.rs_days` counts recent periods where relative strength is positive while the benchmark is in a correction. A correction is defined as the benchmark falling below a configurable fraction of its rolling high.

This example counts AAPL's positive relative-strength days while SPY is at least 5% below its 60-session high.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
strength = q.indicator.relative_strength(prices["AAPL"], prices["SPY"], lookback=21)
strength.tail()

datetime
2026-07-13    0.052857
2026-07-14    0.043351
2026-07-15    0.104698
2026-07-16    0.127166
2026-07-17    0.122123
Name: relative_strength, dtype: float64

## Calculate the indicator

The relative-strength series and SPY prices are explicit inputs. `correction_threshold=0.95` means 5% below the rolling high.

In [2]:
window = 60
threshold = 0.95
days = q.indicator.rs_days(
    strength,
    prices["SPY"],
    window=window,
    correction_threshold=threshold,
)
days.tail()

datetime
2026-07-13    0.0
2026-07-14    0.0
2026-07-15    0.0
2026-07-16    0.0
2026-07-17    0.0
Name: rs_days, dtype: float64

## Compare with SPY

SPY drawdown shows when the correction condition can be active. The count rises only when AAPL is outperforming during those correction dates.

In [3]:
spy_drawdown = prices["SPY"].div(prices["SPY"].rolling(window).max()).sub(1).mul(100)
comparison = pd.DataFrame(
    {
        "AAPL positive RS days": days,
        "SPY drawdown (%)": spy_drawdown,
        "Correction threshold (%)": (threshold - 1) * 100,
    }
)
fig = q.plot.col(
    comparison,
    title="AAPL relative-strength days during SPY corrections",
    ylabel="Sessions / percent",
)
fig.show(renderer="notebook_connected")